In [0]:
spark.read.csv('/Volumes/fraude_proyecto/bronce/bronce_crudo/train_transaction.csv', header=True, inferSchema=True).show(5)

In [0]:
ruta = "/Volumes/fraude_proyecto/bronce/bronce_crudo/"

df_trans = spark.read.csv(ruta + "train_transaction.csv", header=True, inferSchema=True)
df_id = spark.read.csv(ruta + "train_identity.csv", header=True, inferSchema=True)

df = df_trans.join(df_id, on="TransactionID", how="left")
print(f"Filas: {df.count()}, Columnas: {len(df.columns)}")

In [0]:
import re
def limpiar_nombre(c):
    return re.sub(r'[^0-9a-zA-Z_]', '_', c).lower()

for c in df.columns:
    df = df.withColumnRenamed(c, limpiar_nombre(c))

print(df.columns[:10])  # para verificar que quedaron en minúscula

In [0]:
from pyspark.sql.functions import col, when

cols_alto_nulo = ["dist2", "d7", "d13", "d14", "d12", "d6", "d9", "d8", "v149", "v153"]
for c in cols_alto_nulo:
    df = df.withColumn(f"{c}_es_nulo", when(col(c).isNull(), 1).otherwise(0))

print("Flags creados:", [c for c in df.columns if c.endswith("_es_nulo")])

In [0]:
from pyspark.ml.feature import Imputer

num_cols = [c for c, t in df.dtypes if t in ("double", "int", "float", "bigint") 
            and c not in ("transactionid", "isfraud")]

imputer = Imputer(inputCols=num_cols, outputCols=num_cols, strategy="median")
df = imputer.fit(df).transform(df)

print(f"Columnas numéricas imputadas: {len(num_cols)}")

In [0]:
from pyspark.sql.functions import count as _count

cat_cols = [c for c, t in df.dtypes if t == "string"]
for c in cat_cols:
    moda_row = (df.groupBy(c).agg(_count("*").alias("n"))
                  .orderBy(col("n").desc())
                  .filter(col(c).isNotNull())
                  .first())
    if moda_row:
        df = df.na.fill({c: moda_row[c]})

print(f"Columnas categóricas imputadas: {len(cat_cols)}")

In [0]:
antes = df.count()
df = df.dropDuplicates(["transactionid"])
print(f"Filas antes: {antes}, después: {df.count()}, duplicados eliminados: {antes - df.count()}")

In [0]:
df.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
  .saveAsTable("fraude_proyecto.silver.fraud_silver")

In [0]:
df_check = spark.read.table("fraude_proyecto.silver.fraud_silver")
print(f"Filas: {df_check.count()}, Columnas: {len(df_check.columns)}")

In [0]:
df_silver = spark.read.table("fraude_proyecto.silver.fraud_silver")

df_silver.groupBy("isfraud").count().show()

In [0]:
df_silver.select("transactionamt", "transactiondt").describe().show()

In [0]:
display(df_silver.select("transactionamt"))

In [0]:
from pyspark.sql.functions import floor, count as _count, sum as _sum

df_temporal = df_silver.withColumn("dia_relativo", floor(col("transactiondt") / 86400))

tendencia = (df_temporal.groupBy("dia_relativo")
             .agg(_count("*").alias("num_transacciones"),
                  _sum("isfraud").alias("num_fraudes"))
             .orderBy("dia_relativo"))

display(tendencia)

In [0]:
display(df_silver.select("isfraud", "transactionamt"))

In [0]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.stat import Correlation

cols_correlacion = ["transactionamt", "c1", "c2", "c13", "c14", "isfraud"]

assembler = VectorAssembler(inputCols=cols_correlacion, outputCol="features")
df_vector = assembler.transform(df_silver.select(cols_correlacion).na.drop())

matriz = Correlation.corr(df_vector, "features").head()[0]
print(matriz.toArray())

In [0]:
from pyspark.sql.functions import when as _when, count as _count, sum as _sum

df_rangos = df_silver.withColumn(
    "rango_monto",
    _when(col("transactionamt") < 50, "0-50")
    .when(col("transactionamt") < 100, "50-100")
    .when(col("transactionamt") < 500, "100-500")
    .otherwise("500+")
)

gold_tasa_fraude_monto = (df_rangos.groupBy("rango_monto")
    .agg(
        _count("*").alias("total_transacciones"),
        _sum("isfraud").alias("total_fraudes"),
        (_sum("isfraud") / _count("*")).alias("tasa_fraude")
    )
    .orderBy("rango_monto"))

gold_tasa_fraude_monto.show()

gold_tasa_fraude_monto.write.format("delta").mode("overwrite") \
    .saveAsTable("fraude_proyecto.gold.tasa_fraude_por_monto")

In [0]:
from pyspark.sql.functions import avg

gold_ticket_producto = (df_silver.groupBy("productcd")
    .agg(
        avg("transactionamt").alias("ticket_promedio"),
        _count("*").alias("num_transacciones"),
        (_sum("isfraud") / _count("*")).alias("tasa_fraude")
    ))

gold_ticket_producto.show()

gold_ticket_producto.write.format("delta").mode("overwrite") \
    .saveAsTable("fraude_proyecto.gold.ticket_por_producto")

In [0]:
gold_fraude_email = (df_silver.groupBy("p_emaildomain")
    .agg(
        _count("*").alias("total_transacciones"),
        _sum("isfraud").alias("total_fraudes"),
        (_sum("isfraud") / _count("*")).alias("tasa_fraude")
    )
    .filter(col("total_transacciones") > 50)
    .orderBy(col("tasa_fraude").desc()))

gold_fraude_email.show()

gold_fraude_email.write.format("delta").mode("overwrite") \
    .saveAsTable("fraude_proyecto.gold.fraude_por_email")

In [0]:
spark.sql("SELECT * FROM fraude_proyecto.gold.tasa_fraude_por_monto").show()
spark.sql("SELECT * FROM fraude_proyecto.gold.ticket_por_producto").show()
spark.sql("SELECT * FROM fraude_proyecto.gold.fraude_por_email").show()